Maybe try to read in the opera datasets into MintPy for improved time series velocities and ascending/descending decomposition? Then clip the datasets by the aquifers for zonal statistics?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from mintpy import view, tsview
import os
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd

from mintpy.cli.tsview import cmd_line_parse
from mintpy.tsview import timeseriesViewer
from mintpy.view import viewer
from mintpy import geocode
from mintpy import info
import h5py

import rioxarray as rxr
import xarray as xr

In [ ]:
gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")


data_storage = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"
mintpy_paths = list(data_storage.glob('*/mintpy'))
opera_product_paths = list(data_storage.glob('*/subset-ncs/*.nc'))

In [ ]:
data_storage / '1092' / 'orbit_data' / 'OPERA_L2_CSLC-S1-STATIC_T005-008737-IW3_20140403_S1A_v1.0.h5'

In [ ]:
test_clsc = h5py.File(data_storage / '1092' / 'orbit_data' / 'OPERA_L2_CSLC-S1-STATIC_T005-008737-IW3_20140403_S1A_v1.0.h5', 'r')

In [ ]:
test_clsc.keys()

In [ ]:
list(test_clsc['data'].keys())

In [ ]:
plt.imshow(list(test_clsc['data']['los_east']))

In [ ]:
list(test_clsc['data']['los_north'])

In [ ]:
list(test_clsc['identification'].keys())

In [ ]:
list(test_clsc['metadata']['orbit'].keys())

In [ ]:
test_clsc['identification']['look_direction']

In [ ]:
rxr.open_rasterio(opera_product_paths[0], masked=True)

In [ ]:
opera_product_paths[0]

In [ ]:
test_data = xr.open_dataset(mintpy_paths[0].parent / f'disp-output-{mintpy_paths[0].parent.stem}.nc')
# test_data = rxr.open_rasterio(mintpy_paths[0].parent / f'disp-output-{mintpy_paths[0].parent.stem}.nc', masked=True)


In [ ]:
test_data

In [ ]:
test = mintpy_paths[-1]

# list the prepared h5 files for use in MintPy
os.listdir(test)

In [ ]:
for file in os.listdir(test):
    display(list(h5py.File(test / file, 'r').keys()))

In [ ]:
test_h5 = h5py.File(test / 'timeseries.h5', 'r')
list(test_h5.keys())

In [ ]:
perp_baseline = test_h5['bperp']
dates = test_h5['date']
ts_ds = test_h5['timeseries']

In [ ]:
perp_baseline[:]

In [ ]:
import numpy as np

np.unique(ts_ds[:][0])

In [ ]:
plt.imshow(ts_ds[:][0])

#### Tentatively, I will be skipping the additional ionospheric correction that may be implemented in MintPy. The needed information for this may not be available(?) 

# Step 1 Decompose Ascending and Descending tracks into horizontal and vertical displacements from current LOS
- asc_des2horz_vert.py

In [ ]:
from mintpy import asc_desc2horz_vert
from mintpy.utils import readfile

In [ ]:
# testing the asc_des2horz_vert on the displacement dataset first
# will then test on the velocity

# TODO: check how to integrate the recommended mask 

# TODO: co-kriging between GNSS and InSAR for filling in masked/incoherent pixels as another project? May havbe to compare NN, linear, non-linear, and ML interpolation methods with co-kriging


ts_atr_list = [readfile.read_attribute(path / 'timeseries.h5') for path in mintpy_paths]
ts_density_atr_list = [readfile.read_attribute(path / 'timeseries_density.h5') for path in mintpy_paths]
vel_atr_list = [readfile.read_attribute(path / 'velocity.h5') for path in mintpy_paths]
mask_atr_list = [readfile.read_attribute(path / 'recommended_mask_90thresh.h5') for path in mintpy_paths]
coh_atr_list = [readfile.read_attribute(path / 'avgSpatialCoh.h5') for path in mintpy_paths]

In [ ]:
mask_atr_list[0]

In [ ]:
asc_desc2horz_vert.get_overlap_lalo(atr_list)

In [ ]:
ts_files = [path / 'timeseries.h5' for path in mintpy_paths]

In [ ]:
asc_desc2horz_vert.run_asc_desc2horz_vert(ts_files)

# Step 2 Mosaick frames together
- image_stitch.py

In [ ]:


        # cmd = 'timeseries.h5 --yx 273 271 --figsize 8 4'
        # inps = cmd_line_parse(cmd.split())
        # obj = timeseriesViewer(inps)
        # obj.open()
        # obj.plot()

# Step 3 Save results to other data formats
- geotiffs, netcdfs, or h5s of average vertical velocity, as well as the timeseries stacks

In [ ]:
cmd = f'view.py {str(test)}/timeseries.h5'
inps = cmd_line_parse(cmd.split()[1:])
obj = viewer()
obj.configure(inps)
obj.plot()

In [ ]:
tsview([str(test / 'velocity.h5')])